#**Data Collection**

In [1]:
# Install library Hugging Face Datasets terlebih dahulu
!pip install --upgrade "datasets<=3.6.0" pandas

import pandas as pd
from datasets import load_dataset

In [2]:
# Menarik Dataset NusaX-Senti (10.000 data teks Bahasa Indonesia)
# Dataset ini sangat bersih untuk melatih model
nusax_dist = load_dataset("indonlp/NusaX-senti", "ind")
df_nusax = pd.DataFrame(nusax_dist['train'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
# Menarik Dataset IndoNLU / EmoT (4.000+ data teks emosi Bahasa Indonesia)
# Dataset tambahan agar variasi emosinya makin kaya untuk Big Five
emot_dist = load_dataset("indonlp/indonlu", "emot")
df_emot = pd.DataFrame(emot_dist['train'])

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import re

# ==========================================
# MENYIAPKAN DATASET
# ==========================================

# Ambil data NusaX yang sudah di-download sebelumnya
df_nusax_bersih = df_nusax[['text']].rename(
    columns={'text': 'teks_mentah'}
)

# Ambil data IndoNLU / EmoT yang sudah di-download sebelumnya
df_emot_bersih = df_emot[['tweet']].rename(
    columns={'tweet': 'teks_mentah'}
)

# Ambil data Random secara mandiri
review = pd.read_excel("/content/drive/MyDrive/Project_NLP/Project_kelompok_UAS/Data/review.xlsx")
kolom = "data3"

# Mengambil kolom 'data3' dan mengubah namanya menjadi 'teks_mentah'
review = review[[kolom]].rename(
    columns={kolom: 'teks_mentah'}
)

# Gabungkan ketiga dataset
df_review = pd.concat([df_nusax_bersih, df_emot_bersih, review], ignore_index=True)
print(f"Total Dataset Gabungan: {len(df_review)} baris data!")

Total Dataset Gabungan: 64821 baris data!


In [6]:
# Cek jumlah total baris kosong di setiap kolom
print("Jumlah baris kosong per kolom:")
print(df_review.isnull().sum())

# Cek persentase baris kosong
print("\nPersentase baris kosong:")
print(df_review.isnull().mean() * 100)

# Menampilkan baris kosong
baris_kosong = df_review[df_review.isna().any(axis=1)]
display(baris_kosong)

Jumlah baris kosong per kolom:
teks_mentah    4
dtype: int64

Persentase baris kosong:
teks_mentah    0.006171
dtype: float64


,teks_mentah
30355,NaN
51520,NaN
56352,NaN
57593,NaN


In [7]:
# Menghapus semua baris yang memiliki setidaknya satu nilai kosong
df_review = df_review.dropna()

In [8]:
# Cek jumlah total baris kosong di setiap kolom
print("Jumlah baris kosong per kolom:")
print(df_review.isnull().sum())

# Cek persentase baris kosong
print("\nPersentase baris kosong:")
print(df_review.isnull().mean() * 100)

# Menampilkan baris kosong
baris_kosong = df_review[df_review.isna().any(axis=1)]

Jumlah baris kosong per kolom:
teks_mentah    0
dtype: int64

Persentase baris kosong:
teks_mentah    0.0
dtype: float64


In [9]:
df_review.head(10)

,teks_mentah
0,Nikmati cicilan 0% hingga 12 bulan untuk pemes...
1,Kue-kue yang disajikan bikin saya bernostalgia...
2,Ibu pernah bekerja di grab indonesia
3,Paling suka banget makan siang di sini ayam sa...
4,Pelayanan bus DAMRI sangat baik
5,Mau bikin postingan yang isinya mengedukasi cu...
6,Ratusan rumah di medan terendam banjir
7,"Barangnya lumayan, cuma yang saya heran xiaomi..."
8,Sulit sekali mempercayai orang yang sudah pern...
9,"Lokasi di gombel dengan pemandangan semarang, ..."


#**Text Preprocessing**

**Case Folding**

Alasan menerapkan case folding (mengubah semua huruf menjadi huruf kecil/ lowercase) sebelum melakukan proses pelabelan AI adalah untuk menghilangkan ambiguitas dan menyamakan persepsi kata pada komputer.

Komputer pada dasarnya melihat teks berdasarkan kode biner atau angka ASCII [berita]. Di mata komputer, huruf besar dan huruf kecil adalah dua entitas yang benar-benar berbeda.

**1. Menghindari Duplikasi Makna Kata (Tokenisasi yang Efisien)**

Netizen Indonesia sering mengetik dengan gaya kapitalisasi yang acak-acakan [berita]. Tanpa case folding, komputer akan menganggap kata-kata berikut sebagai kata yang berbeda:

* "Stres" (Kapital di awal)
* "stres" (Huruf kecil semua)
* "STRES" (Huruf besar semua)

Jika langsung memasukkannya ke model AI, model harus bekerja 3 kali lebih keras untuk mempelajari bahwa ketiga variasi tersebut memiliki makna psikologis yang sama (Neuroticism). Dengan case folding, semua diseragamkan menjadi "stres", sehingga performa model menjadi jauh lebih ringan dan akurat.

**2. Memaksimalkan Kecocokan Kamus (Kamus Vocabulary) Model AI**

Model dasar seperti DeBERTa-v3 yang digunakan untuk pelabelan, atau IndoRoBERTa yang akan dilatih nanti, memiliki daftar kata terbatas di dalam otaknya (Vocabulary List).

* Mayoritas kata dasar emosi di dalam kamus AI disimpan dalam format huruf kecil.
* Jika AI mendeteksi kata "DEADLINE", ia mungkin akan bingung atau memotongnya menjadi serpihan kecil yang tidak berarti. Namun jika diubah menjadi "deadline", AI akan langsung mengenalinya secara instan dan mengaitkannya dengan sifat kepribadian tertentu secara tepat.

**3. Mengurangi Bias Gaya Mengetik Netizen**

Di media sosial, orang yang sedang marah atau panik cenderung mengetik dengan huruf kapital ("AKU PUSING BANGET!!"). Sebaliknya, orang yang santai mengetik dengan huruf kecil.

* Kita ingin model AI kita berfokus pada arti kata/konteksnya, bukan pada bentuk visual hurufnya.
* Case folding bertindak sebagai "penyetara" yang adil agar emosi murni di dalam teks bisa diekstrak tanpa terdistraksi oleh gaya penulisan caps lock.

In [10]:
import re

def teks_clean(df_review, teks_mentah):

    # Ubah data menjadi huruf kecil
    df_review['teks_bersih'] = df_review[teks_mentah].str.lower()

    # Hapus spasi ganda atau baris kosong
    df_review['teks_bersih'] = df_review['teks_bersih'].str.strip()

    # Pastikan semua data di kolom teks adalah String murni
    df_review['teks_bersih'] = df_review['teks_bersih'].astype(str)

    # Buang baris yang mendadak kosong atau hanya berisi spasi
    df_review = df_review[df_review['teks_bersih'] != ''].reset_index(drop=True)
    df_review = df_review[df_review['teks_bersih'] != 'nan'].reset_index(drop=True)

    # Hapus tanda baca
    df_review['teks_bersih'] = df_review['teks_bersih'].str.replace(r'[^\w\s]', '', regex=True)

    # Hapus huruf yang berulang dengan menggunakan re.sub untuk keandalan regex
    df_review['teks_bersih'] = df_review['teks_bersih'].apply(lambda x: re.sub(r'(.)\1{2,}', r'\1', str(x)))

    # Hapus baris yang teks_bersih kosong/NaN (jika ada)
    df_review = df_review.dropna(subset=['teks_bersih']).reset_index(drop=True)
    df_review['teks_bersih'] = df_review['teks_bersih']

    return df_review

In [11]:
# Jalankan fungsi
df_review = teks_clean(df_review, "teks_mentah")

print(df_review[['teks_mentah', 'teks_bersih']])

                                             teks_mentah  \
0      Nikmati cicilan 0% hingga 12 bulan untuk pemes...   
1      Kue-kue yang disajikan bikin saya bernostalgia...   
2                   Ibu pernah bekerja di grab indonesia   
3      Paling suka banget makan siang di sini ayam sa...   
4                        Pelayanan bus DAMRI sangat baik   
...                                                  ...   
64810  makasih banyak ka' setelah ini aku bakal minta...   
64811  bisa2nya aku baru nonton video ini, makasih ba...   
64812  kak km enak banget klo jelasin jd gampang paha...   
64813  Nemu video ini pas aku ngerasain situasi nya, ...   
64814                                   Kumisan sama nih   

                                             teks_bersih  
0      nikmati cicilan 0 hingga 12 bulan untuk pemesa...  
1      kuekue yang disajikan bikin saya bernostalgia ...  
2                   ibu pernah bekerja di grab indonesia  
3      paling suka banget makan siang di si

In [12]:
# Merubah nama dataset dan mengambil kolom "teks_bersih"
df_master=df_review[['teks_bersih']]

df_master

,teks_bersih
0,nikmati cicilan 0 hingga 12 bulan untuk pemesa...
1,kuekue yang disajikan bikin saya bernostalgia ...
2,ibu pernah bekerja di grab indonesia
3,paling suka banget makan siang di sini ayam sa...
4,pelayanan bus damri sangat baik
...,...
64810,makasih banyak ka setelah ini aku bakal minta ...
64811,bisa2nya aku baru nonton video ini makasih ban...
64812,kak km enak banget klo jelasin jd gampang paha...
64813,nemu video ini pas aku ngerasain situasi nya t...


**Zero-Shot Classification (Pelabelan Otomatis dari Tweet ke 5 Big Five Personality Lewat AI)**

Sebelum teknologi ini ada, jika punya 13.000 data acak yang tidak ada labelnya, kita harus melakukan Manual Labeling (membaca dan mengetik satu per satu) atau melatih model klasifikasi baru dari nol (Supervised Learning). Namun, Supervised Learning butuh puluhan ribu data yang sudah ada kunci jawabannya.

* Kemampuan NLI (Natural Language Inference): Metode ini bekerja dengan cara membandingkan kalimat curhatan user (Premise) dengan kalimat hipotesis (Hypothesis).
* Cara AI Berpikir: AI akan mengetes: "Jika user berkata 'aku pusing dikerjar deadline', apakah kalimat ini bermakna dekat dengan hipotesis 'Teks ini berbicara tentang Neuroticism'?"
* Kelebihan Utama: Bisa langsung memberikan label abstrak seperti Neuroticism atau Extraversion kepada data baru tanpa perlu melatih AI tersebut terlebih dahulu. Akurasinya sangat tinggi karena AI memahami esensi bahasa, bukan sekadar mencocokkan kata kunci.

**AI Transformer DeBERTa-v3**

Memilih DeBERTa-v3 (dikembangkan oleh Microsoft) karena model ini adalah rajanya akurasi untuk tugas pemahaman teks (Natural Language Understanding)

* Mekanisme Disentangled Attention (Perhatian Terurai): Tidak seperti BERT biasa yang melihat kata secara kaku, DeBERTa melihat kata berdasarkan dua hal terpisah: arti kata dan posisi kata dalam kalimat. Hal ini membuatnya sangat peka terhadap susunan kalimat curhat netizen yang polanya acak-acakan.
* Enhanced Masked Language Model (EMLM): Saat DeBERTa dilatih di pabriknya, dia diajarkan untuk memprediksi kata yang hilang bukan hanya berdasarkan kata di sekitarnya, tapi juga berdasarkan struktur tata bahasanya secara global.
* Akurasi Terbaik di Kelasnya: Di papan peringkat sains (Benchmark SuperGLUE), DeBERTa-v3 berhasil mengalahkan performa manusia dan model-model besar lainnya dalam memahami logika bahasa.

In [13]:
# Install library yang dibutuhkan
!pip install transformers[torch] pandas openpyxl

from transformers import pipeline
from tqdm import tqdm

# Inisialisasi Model Zero-Shot Text Classification (Gratis & Sangat Akurat)
# Model ini dilatih khusus untuk memahami hubungan logika antar-kalimat
classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    device=0 # PENTING: Angka 0 artinya menyuruh AI berjalan di atas T4 GPU Colab
)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

**Teori Big Five Personality (Lima Sifat Dasar)**

* **Openness (Keterbukaan):** Tingkat imajinasi, rasa ingin tahu, dan kreativitas seseorang.
* **Conscientiousness (Kesadaran):** Sifat disiplin, terorganisir, dan tanggung jawab yang tinggi.
* **Extraversion (Ekstraversi):** Tingkat energi dan kenyamanan bersosialisasi dengan lingkungan luar.
* **Agreeableness (Persetujuan):** Sifat penuh kasih sayang, empati, dan kemampuan bekerja sama.
* **Neuroticism (Neurotisisme):** Kecenderungan untuk mengalami emosi negatif seperti kecemasan atau perubahan suasana hati.

In [14]:
# 5 Kategori Sifat Utama (Big Five Personality) sebagai kandidat untuk melatih model AI
kandidat_ocean = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]

print(f"Menyiapkan pelabelan berbasis konteks untuk {len(df_master)} baris data...")

Menyiapkan pelabelan berbasis konteks untuk 64815 baris data...


In [15]:
# Proses Pelabelan Otomatis (Kita gunakan progress bar tqdm agar perkembangannya terlihat)
list_label_final = []

# Catatan Taktis: Karena proses AI membaca 17ribu data akan memakan waktu cukup lama
# Di bawah ini sistem diset memproses seluruh data.
for teks in tqdm(df_master['teks_bersih'], desc="Memproses Teks"):
    try:
        # AI memprediksi persentase kecocokan teks dengan 5 sifat OCEAN
        hasil = classifier(str(teks), candidate_labels=kandidat_ocean)
        # Ambil label dengan skor probabilitas tertinggi (Pemenang Utama)
        label_terbaik = hasil['labels'][0]
        list_label_final.append(label_terbaik)
    except Exception as e:
        # Antrean cadangan jika ada teks kosong/eror mendadak
        list_label_final.append("Openness")

Memproses Teks: 100%|██████████| 64815/64815 [2:34:37<00:00,  6.99it/s]


In [19]:
# Memasukkan hasil tebakan AI yang akurat ke dalam kolom baru
df_master['sifat'] = list_label_final

In [20]:
# Simpan hasil akhir Babak 1 yang super akurat ini
df_master.to_csv("master_dataset.csv", index=False)

print("\n🎉 PROSES ZERO-SHOT LABELING SELESAI 100%!")
print(f"📉 Dataset final kelompok kita kini memiliki kunci jawaban yang valid!")
print("\n📋 Distribusi hasil label psikologi buatan DeBERTa AI:")
print(df_master['label_ocean'].value_counts())
print("\n👉 File 'master_dataset_berlabel_ai.csv' siap digunakan untuk melatih IndoRoBERTa!")


🎉 PROSES ZERO-SHOT LABELING SELESAI 100%!
📉 Dataset final kelompok kita kini memiliki kunci jawaban yang valid!

📋 Distribusi hasil label psikologi buatan DeBERTa AI:
label_ocean
Extraversion         26575
Openness             15189
Agreeableness        13770
Conscientiousness     5196
Neuroticism           4085
Name: count, dtype: int64

👉 File 'master_dataset_berlabel_ai.csv' siap digunakan untuk melatih IndoRoBERTa!
